In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Загрузка данных**

In [20]:
file_path = 'exel_files/amazon_delivery.csv'
df = pd.read_csv(file_path)

**Препроцессинг и Feature Engineering**

In [21]:
# Переименование колонок для удобства
df.columns = ['Order_ID', 'Age', 'Rating', 'Delivery_Lat', 'Delivery_Long', 
              'Store_Lat', 'Store_Long', 'Order_Date', 'Order_Time', 
              'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 
              'Area', 'Time_Taken_min', 'Category']

# Удаление пропущенных значений
df.replace('NaN', np.nan, inplace=True)
df.dropna(inplace=True)

**Расчет целевой переменной (Time_Taken_min) и очистка**

In [22]:
df['Time_Taken_min'] = pd.to_numeric(df['Time_Taken_min'], errors='coerce')
df.dropna(subset=['Time_Taken_min'], inplace=True)

**Расчет расстояния (Haversine distance)**

In [24]:
from geopy.distance import geodesic

def calculate_distance(row):
    try:
        start_point = (row['Store_Lat'], row['Store_Long'])
        end_point = (row['Delivery_Lat'], row['Delivery_Long'])
        return geodesic(start_point, end_point).km
    except ValueError:
        return np.nan
    
df['Distance_km'] = df.apply(calculate_distance, axis=1)
df.dropna(subset=['Distance_km'], inplace=True)

**Преобразование времени и даты**

In [25]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Hour'] = pd.to_datetime(df['Order_Date']).dt.hour
df['Pickup_Time'] = pd.to_datetime(df['Pickup_Time']).dt.hour
df['Day_of_Week'] = df['Order_Date'].dt.dayofweek # 0 =Пн, 6=Вс
df['Month'] = df['Order_Date'].dt.month

C:\Users\Darin_btw\AppData\Local\Temp\ipykernel_19160\3533200915.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Pickup_Time'] = pd.to_datetime(df['Pickup_Time']).dt.hour


In [26]:
#Кодирование категориальных признаков
categorical_cols = ['Weather', 'Traffic', 'Vehicle', 'Area', 'Category']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [27]:
features = [col for col in df.columns if col not in[
    'Order_ID', 'Order_Date', 'Order_Time', 'Pickup_Time',
    'Delivery_Lat', 'Delivery_Long', 'Store_Lat', 'Store_Long',
    'Time_Taken_min'
]]
target = 'Time_Taken_min'

X = df[features]
y = df[target]

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
import os
# Визуализация
output_dir = 'images'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df[target], kde=True, bins=30)
plt.title('Распределение времени доставки (Time_Taken_min)')
plt.xlabel('Время доставки (мин)')
plt.ylabel('Частота')
plt.savefig(os.path.join(output_dir, 'target_distribution.png'))
plt.close()

IndexError: Inconsistent shape between the condition and the input (got (43594, 1) and (43594,))

<Figure size 1000x600 with 0 Axes>